In [1]:
!pip install -U transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 101.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 34.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Foun

# dataset
### we are going to use `Pycoder156/hr_policy_asst_non_instruction_dataset_v0` dataset

In [2]:
from datasets import load_dataset, Dataset
dataset = load_dataset('Pycoder156/hr_policy_asst_non_instruction_dataset_v0')
dataset

README.md:   0%|          | 0.00/495 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.7kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.29kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/95 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 95
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 17
    })
})

In [3]:
dataset['train'][0]

{'text': 'Successful graduation from a Performance Improvement Plan requires the employee to hit every operational target specified in the corrective document. Conversely, if the individual fails to demonstrate significant, sustained improvement by the end of the PIP window, immediate separation protocols are triggered. Under these circumstances, the company may opt for immediate termination based on performance, bypassing standard notice structures where legally permissible.',
 'source_page': 9,
 'paragraph_id': 2,
 'char_count': 470}

# base model for fine turning

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [74]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

LlamaTokenizer(name_or_path='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', vocab_size=32000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [75]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'</s>'

In [76]:
def tokenizer_fn(data):
    tokens = tokenizer(data['text'],truncation=True,max_length=512,padding='max_length')
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

In [77]:
dataset = dataset.map(tokenizer_fn,batched=True,remove_columns=['text','source_page','paragraph_id','char_count'])
dataset

ValueError: Column to remove ['source_page', 'text', 'char_count', 'paragraph_id'] not in the dataset. Current columns in the dataset: ['input_ids', 'attention_mask', 'labels']

## using quantization bcs we have limited GPU resource

In [78]:
from peft import TaskType, LoraConfig, get_peft_model, PeftModel
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bnb_config,
                                             device_map='auto',
                                            dtype=torch.bfloat16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# lora config for fine-turning

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM)

lora_model = get_peft_model(model,lora_config)
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [11]:
train_args = TrainingArguments(
    num_train_epochs=35,
    output_dir='fine-turning-output',
    save_steps=10,
    save_total_limit=2,
    logging_steps=1,
    per_device_train_batch_size=10,
    gradient_accumulation_steps=8, # Increased gradient accumulation
    gradient_checkpointing=True, # Added gradient checkpointing
)

trainer = Trainer(model=lora_model,args=train_args,train_dataset=dataset['train'],eval_dataset=dataset['validation'])

## to clear unwanted logging and warning

In [12]:
import logging
# Mute the specific bitsandbytes internal logger module
logger = logging.getLogger("bitsandbytes.autograd._functions")
logger.setLevel(logging.ERROR)

## my GPU is full so i train with checkpoint then clear GPU memory, again repeat step

In [13]:
trainer.train(resume_from_checkpoint="/kaggle/working/fine-turning-output/checkpoint-60")

ValueError: Can't find a valid checkpoint at /kaggle/working/fine-turning-output/checkpoint-60

In [ ]:
trainer.save_model('fine-turned-save-6')

## inferance from fine-turned adapters

In [79]:
adapter_path = "/kaggle/input/models/strangerias/fine-turned-adapter/transformers/default/1"
ft_model = PeftModel.from_pretrained(model,adapter_path)
ft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [124]:
ft_model.eval()
eval_prompts = [
    # 🔒 1. Data Security & Confidentiality Policies
    "Sharing company internal source code or confidential intellectual property on public forums will result in",
    "Employees must ensure that all customer personal data is encrypted and stored only on",
    "Under no circumstances shall an employee disclose proprietary company algorithms or trade secrets to",
    "To protect corporate network integrity, downloading unauthorized third-party software onto work devices is",
    
    # ⚖️ 2. Indemnity, Liability & Legal Boilerplate
    "You agree to indemnify, defend and hold harmless the Company, its officers, directors, employees, and agents from",
    "Any violation of these corporate terms or negligent conduct on company networks will result in",
    "The Company's total liability to any employee or third party for damages arising out of",
    "These terms and policies shall be governed by and construed in accordance with the laws of",
    
    # 🌐 3. Digital Asset & Device Usage Policies
    "In an attempt to provide increased value, the Company may link to sites operated by third parties; however,",
    "Employees access external links and third-party partner portals entirely at their own",
    "The corporate network and company-issued laptops must be used strictly for",
    "Monitoring of company email accounts and internal communication channels may be conducted by IT tools for the purpose of",
    
    # 👥 4. Workplace Conduct & HR Operations
    "The Company reserves the right, at its sole discretion, to modify, amend, or terminate any clause in this handbook at",
    "An employee who wishes to dispute a performance review or disciplinary action must submit a formal appeal to",
    "Standard working hours and remote work flexibility options are subject to approval by",
    "In the event that an employee fails to adhere to the standard code of conduct, the disciplinary framework involves"
]

def ft_inferance(prompt, model, tokenizer):
    input_prompt_tokenized = tokenizer(prompt,return_tensors='pt').to('cuda')
    model_response = ft_model.generate(**input_prompt_tokenized,
                                       max_new_tokens=256,
                                      # 1. Break the deterministic loop by enabling sampling
                                        do_sample=True,           
                                        temperature=0.7,          # Lower (e.g., 0.4) for rigid facts, higher (0.7-0.9) for flow
                                        top_p=0.9,                # Keeps only the most confident tokens totaling 90% probability
                                       # 2. Prevent repetition
                                        repetition_penalty=1.2,   # Penalizes the model heavily if it repeats exact phrases
                                        no_repeat_ngram_size=3,   # Changes the probability of any 3-word phrase repeating to 0
                                        
                                        # 3. Handle stopping properly
                                        pad_token_id=tokenizer.eos_token_id,
                                        eos_token_id=tokenizer.eos_token_id
                                        )
    output = tokenizer.decode(model_response[0])
    return output

inferance = {}


In [127]:
import json
for prompt_id in range(0,1):
    prompt = eval_prompts[prompt_id]
    prompt_inferance = ft_inferance(prompt,ft_model,tokenizer)

    ## saving inferance with json for eval
    # read json file
    with open('inferance.json','r') as file:
        file_data = json.load(file)
        file_data[prompt_id] = {"question": prompt,
                           "model_response": prompt_inferance}
    
    with open('inferance.json','w') as file:
        json.dump(file_data,file)
        
    print(prompt)
    print("=="*30)
    print(prompt_inferance)
    print("=="*30)
    

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sharing company internal source code or confidential intellectual property on public forums will result in
<s> Sharing company internal source code or confidential intellectual property on public forums will result in immediate termination of this license.
10.3 You may not export or re-export any Software, other than to the countries where such software is permitted under applicable law. In particular, you may not download and/or otherwise export or transfer any Software that has been downloaded from the website if it contains (a) a technology controlled by an entity located in Iran, Iraq, North Korea, Sudan or Syria; or (b) a technological solution that requires access to or use of an element that is subject to U.S. embargoes. By downloading and using our Software, you represent and warrant to us that you are not located in, under the control of, or a national or resident of any country prohibited from receiving the Software under U.N. sanctions programs. If we discover at a later dat

In [121]:
with open('/kaggle/working/inferance.json','w') as file:
    json.dump({},file)

In [128]:
with open('/kaggle/working/inferance.json','r') as file:
    data = json.load(file)
data

{'1': {'question': 'Employees must ensure that all customer personal data is encrypted and stored only on',
  'model_response': '<s> Employees must ensure that all customer personal data is encrypted and stored only on servers in the UK.\nGDPR – GDPR: Information about how we use your information, including any of the following activities; (i) contacting you by email or telephone; (ii) sending postal mail to you using third party services such as MailChimp, Constant Contact etc.; (iii) providing surveys, market research, statistical analysis and other communications related to our products, programs or services.\nPurposes for which the information will be used include: processing orders, payments, account management, support, promotions, offers, newsletters, surveys and/or competitions; responding to requests from customers via e-mails and phone calls; and managing complaints, enquiries and queries.\nWe may transfer personal data outside the European Economic Area ("EEA") where necessa